# Chapter 7: Working with Text Data
## *Introduction to Machine Learning with Python*  Müller & Guido

---
> **Ringkasan Bab:** Teks adalah salah satu jenis data yang paling melimpah di dunia. Bab ini membahas cara merepresentasikan teks sebagai fitur numerik (Bag of Words, TF-IDF), klasifikasi teks (analisis sentimen), dan topik model (LDA).

## 7.1 Representasi Teks sebagai Angka

ML tidak bisa langsung memproses teks  kita perlu mengubahnya menjadi **vektor numerik**.

**Pipeline NLP dasar:**
```
Teks mentah → Tokenisasi → [Stop words removal] → Vektorisasi → Model ML
```

**Dua pendekatan utama:**

| Metode | Deskripsi | Kelebihan |
|---|---|---|
| **Bag of Words (BoW)** | Hitung frekuensi tiap kata | Sederhana, efektif |
| **TF-IDF** | Bobot kata berdasarkan keunikannya | Lebih informatif dari BoW |

## 7.2 Bag of Words (CountVectorizer)

**Bag of Words (BoW):**
1. Bangun **kosa kata** (vocabulary) dari seluruh corpus
2. Setiap dokumen direpresentasikan sebagai **vektor frekuensi** kata

**Kekurangan:** Tidak mempertimbangkan urutan kata, konteks, atau makna.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Contoh corpus sederhana
corpus = [
    "Saya suka belajar machine learning",
    "Machine learning sangat menarik dan berguna",
    "Belajar Python untuk data science",
    "Data science dan machine learning saling berkaitan",
]

# Vektorisasi dengan CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(corpus)

print("Vocabulary (kata → index):")
print(sorted(vectorizer.vocabulary_.items(), key=lambda x: x[1]))

print("\nMatrix Bag of Words:")
print("(baris = dokumen, kolom = kata)")
df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)
df_bow.index = [f'Dok {i+1}' for i in range(len(corpus))]
print(df_bow)

In [ ]:
# Visualisasi BoW
plt.figure(figsize=(12, 4))
plt.imshow(X_bow.toarray(), cmap='Blues', aspect='auto')
plt.colorbar(label='Frekuensi')
plt.xticks(range(len(vectorizer.get_feature_names_out())),
           vectorizer.get_feature_names_out(), rotation=45, ha='right')
plt.yticks(range(len(corpus)), [f'Dok {i+1}' for i in range(len(corpus))])
plt.title('Bag of Words Matrix')
plt.tight_layout()
plt.show()

## 7.3 TF-IDF (Term Frequency - Inverse Document Frequency)

**TF-IDF** memberi bobot lebih tinggi pada kata yang:
- Sering muncul dalam **satu dokumen** (TF tinggi)
- Jarang muncul di **dokumen lain** (IDF tinggi)

**Formula:**
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\left(\frac{N}{\text{df}(t)}\right)$$

- TF(t, d) = frekuensi kata t dalam dokumen d
- N = jumlah total dokumen
- df(t) = jumlah dokumen yang mengandung kata t

**Intuisi:** Kata seperti "dan", "yang", "di" muncul di semua dokumen → IDF rendah → TF-IDF rendah → kurang informatif.

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=tfidf.get_feature_names_out()
)
df_tfidf.index = [f'Dok {i+1}' for i in range(len(corpus))]

print("TF-IDF Matrix:")
print(df_tfidf)

print("\nPerbandingan BoW vs TF-IDF untuk kata 'machine':")
print(f"BoW   : {df_bow['machine'].values}")
print(f"TF-IDF: {df_tfidf['machine'].values.round(3)}")
print("\n'machine' muncul di banyak dokumen → TF-IDF lebih rendah (kurang diskriminatif)")

## 7.4 N-Gram: Menangkap Konteks

**N-gram** mempertimbangkan urutan kata:
- **Unigram (1-gram):** tiap kata sendiri-sendiri — `['machine', 'learning']`
- **Bigram (2-gram):** pasangan kata — `['machine learning', ...]`
- **Trigram (3-gram):** tiga kata — `['machine learning sangat', ...]`

**Manfaat:** N-gram menangkap frasa penting seperti "tidak baik" vs "baik".

In [ ]:
# Bigram
bigram = CountVectorizer(ngram_range=(1, 2))  # unigram + bigram
X_bigram = bigram.fit_transform(corpus)

print(f"Unigram saja : {len(vectorizer.vocabulary_)} fitur")
print(f"Unigram+Bigram: {len(bigram.vocabulary_)} fitur")
print("\nContoh bigram:", [f for f in bigram.get_feature_names_out() if ' ' in f])

## 7.5 Analisis Sentimen: Text Classification

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Dataset 20 Newsgroups (4 kategori)
categories = [
    'alt.atheism',
    'talk.religion.misc',
    'comp.graphics',
    'sci.space'
]

train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers', 'footers', 'quotes'))

print(f"Jumlah dokumen training: {len(train_data.data)}")
print(f"Jumlah dokumen test    : {len(test_data.data)}")
print(f"Kategori               : {train_data.target_names}")
print(f"\nContoh dokumen:")
print(train_data.data[0][:300], "...")

In [ ]:
# Pipeline: TF-IDF → Logistic Regression
text_pipe = make_pipeline(
    TfidfVectorizer(min_df=5, max_df=0.95, stop_words='english', ngram_range=(1,2)),
    LogisticRegression(max_iter=1000, C=1.0)
)

text_pipe.fit(train_data.data, train_data.target)

y_pred = text_pipe.predict(test_data.data)
acc = (y_pred == test_data.target).mean()
print(f"Akurasi Klasifikasi Teks: {acc:.3f}\n")

print("Classification Report:")
print(classification_report(test_data.target, y_pred,
      target_names=test_data.target_names))

In [ ]:
# Kata-kata paling informatif per kategori
vectorizer_fitted = text_pipe.named_steps['tfidfvectorizer']
clf = text_pipe.named_steps['logisticregression']

feature_names = vectorizer_fitted.get_feature_names_out()

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, (ax, category) in enumerate(zip(axes, categories)):
    top_n = 10
    coef = clf.coef_[i]
    top_pos = np.argsort(coef)[-top_n:][::-1]
    
    ax.barh(range(top_n), coef[top_pos], color='steelblue')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[j] for j in top_pos], fontsize=8)
    ax.set_title(category.split('.')[-1], fontsize=10)
    ax.set_xlabel('Koefisien')

plt.suptitle('Kata Paling Informatif per Kategori', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7.6 Contoh: Analisis Sentimen Sederhana

In [ ]:
# Dataset sentimen sederhana (bahasa Indonesia)
ulasan = [
    "Produk ini sangat bagus dan berkualitas tinggi",
    "Pengiriman cepat, saya sangat puas",
    "Produk rusak saat diterima, sangat kecewa",
    "Tidak sesuai deskripsi, mengecewakan sekali",
    "Kualitas excellent, akan beli lagi",
    "Pelayanan ramah dan responsif, top!",
    "Barang palsu, jangan beli di sini!",
    "Sangat recommended, harga terjangkau kualitas bagus",
    "Paket tidak sampai, penjual tidak respon",
    "Produk original, sesuai ekspektasi, puas banget",
]
# 0 = Negatif, 1 = Positif
sentimen = [1, 1, 0, 0, 1, 1, 0, 1, 0, 1]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    ulasan, sentimen, test_size=0.3, random_state=42
)

# Pipeline sentimen
sentimen_pipe = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    LogisticRegression(max_iter=1000)
)
sentimen_pipe.fit(X_train_s, y_train_s)

print("=== Prediksi Sentimen ===")
ulasan_baru = [
    "Barang bagus, pengiriman cepat!",
    "Kecewa banget, produk tidak sesuai",
    "Sangat puas, akan order lagi",
]

pred = sentimen_pipe.predict(ulasan_baru)
proba = sentimen_pipe.predict_proba(ulasan_baru)

for ulasan_test, p, prob in zip(ulasan_baru, pred, proba):
    label = ' Positif' if p == 1 else ' Negatif'
    print(f"\n'{ulasan_test}'")
    print(f"  → {label} (Prob: Neg={prob[0]:.2f}, Pos={prob[1]:.2f})")

## 7.7 Preprocessing Teks Lanjutan

In [ ]:
# Parameter penting CountVectorizer / TfidfVectorizer

print("=== Parameter Penting Vectorizer ===")
print()

params_explanation = {
    'min_df'      : 'Abaikan kata yang muncul di < min_df dokumen (hapus kata sangat jarang)',
    'max_df'      : 'Abaikan kata yang muncul di > max_df dokumen (hapus stop words umum)',
    'max_features': 'Gunakan hanya N kata paling sering',
    'ngram_range' : 'Rentang n-gram, misal (1,2) = unigram + bigram',
    'stop_words'  : 'Hapus kata umum (english/custom list)',
    'analyzer'    : 'Tokenisasi per kata (word) atau karakter (char)',
    'sublinear_tf': '[TF-IDF] Gunakan log(TF) untuk mengurangi pengaruh kata sangat sering',
}

for param, desc in params_explanation.items():
    print(f"  {param:<16}: {desc}")

print()

# Contoh dengan berbagai konfigurasi
configs = [
    {"name": "Default",          "params": {}},
    {"name": "min_df=2",         "params": {"min_df": 2}},
    {"name": "max_features=100", "params": {"max_features": 100}},
    {"name": "bigram",           "params": {"ngram_range": (1, 2)}},
]

for cfg in configs:
    vect = TfidfVectorizer(**cfg["params"])
    X_v  = vect.fit_transform(train_data.data)
    print(f"  {cfg['name']:<20}: {X_v.shape[1]:>6} fitur | Shape: {X_v.shape}")

## 7.8 Latent Dirichlet Allocation (LDA)  Topic Modeling

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# LDA butuh CountVectorizer (bukan TF-IDF)
cv_lda = CountVectorizer(max_features=1000, stop_words='english', min_df=5)
X_lda  = cv_lda.fit_transform(train_data.data)

# Fit LDA dengan 4 topik
lda = LatentDirichletAllocation(n_components=4, random_state=42, max_iter=20)
lda.fit(X_lda)

# Tampilkan kata-kata teratas tiap topik
feature_names_lda = cv_lda.get_feature_names_out()
print("=== Topik yang Ditemukan LDA ===")
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names_lda[i] for i in topic.argsort()[-10:][::-1]]
    print(f"\nTopik {topic_idx+1}: {' | '.join(top_words)}")

## Ringkasan Bab 7

| Konsep | Penjelasan |
|---|---|
| **Bag of Words** | Representasi teks sebagai frekuensi kata — sederhana |
| **TF-IDF** | Bobot kata berdasarkan kelangkaan di corpus — lebih informatif |
| **N-Gram** | Menangkap frasa multi-kata untuk konteks lebih baik |
| **CountVectorizer** | Implementasi BoW di scikit-learn |
| **TfidfVectorizer** | Implementasi TF-IDF di scikit-learn |
| **Pipeline Teks** | TfidfVectorizer + Classifier = pipeline NLP standar |
| **LDA** | Unsupervised topic modeling |

**Tip Praktis:**
- Mulai dengan `TfidfVectorizer` + `LogisticRegression` sebagai baseline
- Gunakan `min_df` dan `max_df` untuk filtering kata yang tidak informatif
- Coba `ngram_range=(1,2)` untuk menangkap frasa penting
- Selalu gunakan Pipeline agar preprocessing aman saat cross-validation

---